# Portfolio construction 2 of 3: how much

Notebooks: [1 hold](portfolio_construction_1_hold.ipynb) · [2 scale](portfolio_construction_2_scale.ipynb) · [3 trade](portfolio_construction_3_trade.ipynb), matching `src/portfolio_construction/{hold,scale,trade}`. In-sample, 1976-2014, rolling-covariance baseline (36 months, 50% shrinkage), primary cost 2 bps per traded notional (`primary_cost_bps`); data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`).

## Goal

Improve the model. Here: is the 10% risk ceiling the right control, and should scaling be a one-sided limit or a vol target?

## Current pipeline

**What we are doing.** Shrink the whole book when it uses too much leverage or risk. It never enlarges it.

**Inputs**
- Signed weights from the hold stage.
- Forecast covariance `Q` from the risk model.
- Config (`config/risk_config.json`): gross cap 3.0, risk ceiling 10%. I found no rationale for either in the repo.

**How we process them** (`scale/limits.py`)
1. `gross = sum of |weight|`.
2. `risk = sqrt(w'Qw)`, the forecast annual volatility of the book.
3. `scale = min(1, 3.0 / gross, 0.10 / risk)`.
4. Multiply every weight by `scale`.

*Example:* gross 4.5 and risk 12% give `min(1, 0.67, 0.83) = 0.67`. All positions shrink by a third, leaving gross 3.0 and risk 8.0%. The gross cap set the scale here, not the ceiling.

**Output.** The target book, passed to the trade stage.

## Implications

- The ceiling is a construction rule. The risk model supplies one number per month, `sqrt(w'Qw)`.
- Its strictness depends on risk-model calibration: realised vol runs about 27% above forecast, so a 10% ceiling is looser than it reads.
- Being one-sided, it cannot lever up in calm months.

## Results and interpretation

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]


from risk_model_study import load_study, rolling_model, run, summarize

from src.backtest.qps_baseline import forecasts_from, qps_components

pd.options.display.float_format = "{:.3f}".format
study = load_study()
settings, base, returns, source, known, meta = study.inputs
model = rolling_model(study)
parts = qps_components(base, settings, meta, model)
targets, covariance, diagnostics = forecasts_from(parts, returns, meta, base)
ledgers = run(study, model)
print(f"{returns.shape[1]} markets; trend target months: {len(targets['trend'])}")

58 markets; trend target months: 468


In [2]:
bound = diagnostics.query("strategy == 'trend' and eligible >= @base['minimum_instruments']")
trend_ledger = ledgers["trend"]
pd.Series(
    {
        "months raw target breaches 3x gross cap": bound.target_gross_limited.mean(),
        "months raw target breaches 10% risk ceiling": bound.target_risk_limited.mean(),
        "mean target gross": bound.target_gross.mean(),
        "mean target forecast vol": bound.target_forecast.mean(),
        "calibration (realised / forecast vol)": summarize(trend_ledger)["calibration"],
    }
).to_frame("trend")

,trend
months raw target breaches 3x gross cap,0.537
months raw target breaches 10% risk ceiling,0.338
mean target gross,2.467
mean target forecast vol,0.083
calibration (realised / forecast vol),1.272


- **Limits bind often:** the raw target breaches the 3x gross cap in about 54% of months and the 10% ceiling in about 34%.
- **Calibration is 1.27:** realised vol is about 27% above forecast (10.4% against 8.0%). Compare edits at matched *realised* risk.

### 2.1 Which limit sets the scale?

Raw book from `construct_targets` with infinite limits.

In [3]:
from dataclasses import replace

from src.backtest.engine import simulate
from src.portfolio_construction import BandPolicy, PortfolioLimits
from src.risk_model import portfolio_risk
from src.transaction_cost_model import LinearBpsCost

raw_targets, raw_cov, _ = forecasts_from(
    replace(parts, limits=PortfolioLimits(np.inf, np.inf)), returns, meta, base
)
raw = raw_targets["trend"]
sample = returns.loc[base["start"] : base["end"]]
GROSS_CAP, CEILING = settings["gross_cap"], settings["portfolio_risk_ceiling"]

book = pd.DataFrame(
    {
        month: {
            "gross": np.abs(raw.loc[month]).sum(),
            "risk": portfolio_risk(raw.loc[month].to_numpy(), raw_cov[month]),
        }
        for month in raw.index
        if (raw.loc[month] != 0).sum() >= base["minimum_instruments"]
    }
).T
book["gross_ratio"] = GROSS_CAP / book.gross  # < 1 means the gross cap binds
book["risk_ratio"] = CEILING / book.risk  # < 1 means the risk ceiling binds
book["scale"] = book[["gross_ratio", "risk_ratio"]].clip(upper=1.0).min(axis=1)
gross_binds, risk_binds = book.gross_ratio < 1, book.risk_ratio < 1
which = np.select(
    [gross_binds & risk_binds, gross_binds, risk_binds],
    ["both", "gross cap only", "risk ceiling only"],
    default="neither",
)
by_case = book.groupby(which).agg(months=("scale", "size"), mean_scale=("scale", "mean"), mean_raw_risk=("risk", "mean"))
by_case["share"] = by_case.months / len(book)
any_binds = gross_binds | risk_binds
tightest = (book.risk_ratio < book.gross_ratio)[any_binds].mean()
print(f"Months where a limit binds: {any_binds.mean():.1%}; the risk ceiling is the tighter one in {tightest:.1%} of those")
print(f"Raw forecast vol: mean {book.risk.mean():.1%}, median {book.risk.median():.1%}, "
      f"months below the ceiling {(~risk_binds).mean():.1%}")
print(f"Months with room to scale UP to the ceiling (raw risk below it AND gross cap not binding): "
      f"{((~risk_binds) & (~gross_binds)).mean():.1%}")
by_case[["share", "months", "mean_scale", "mean_raw_risk"]]

Months where a limit binds: 76.7%; the risk ceiling is the tighter one in 37.2% of those
Raw forecast vol: mean 9.5%, median 9.2%, months below the ceiling 66.2%
Months with room to scale UP to the ceiling (raw risk below it AND gross cap not binding): 23.3%


,share,months,mean_scale,mean_raw_risk
both,0.109,51,0.793,0.117
gross cap only,0.428,200,0.846,0.082
neither,0.233,109,1.000,0.088
risk ceiling only,0.229,107,0.877,0.115


- A limit binds in **77%** of months: gross cap 54%, ceiling 34%, both 11%. Where one binds, the ceiling is the tighter in only **37%**.
- Raw forecast vol averages 9.5% (median 9.2%), so the ceiling clips the upper third. Mean forecast after limits: 8.0%.
- Only 23% of months have room to scale up under both limits.

### 2.2 Limit versus vol target

Limit: `min(1, g/gross, c/risk)`. Target: `min(g/gross, c/risk)`, which also levers up. Same band, costs and executed limits. A third arm loosens the gross cap to 6x.

In [4]:
def evaluate(ceiling, lever_up, gross_cap=GROSS_CAP):
    scale = (gross_cap / book.gross).combine(ceiling / book.risk, min)
    if not lever_up:
        scale = scale.clip(upper=1.0)
    targets_ = raw.mul(scale.reindex(raw.index).fillna(0.0), axis=0)
    policy = BandPolicy(raw_cov, PortfolioLimits(gross_cap, ceiling), settings["primary_band"])
    ledger, _ = simulate(
        sample, targets_, LinearBpsCost(settings["primary_cost_bps"]),
        source.loc[sample.index], position_policy=policy, source_known_at=known.loc[sample.index],
    )
    row = summarize(ledger)
    row["share_levered_up"] = (scale > 1 + 1e-12).mean()
    row["mean_gross"] = ledger.gross_exposure.mean()
    return row

arms = {
    "limit (gross 3x)": {"lever_up": False},
    "target (gross 3x)": {"lever_up": True},
    "target (gross 6x)": {"lever_up": True, "gross_cap": 6.0},
}
grid = pd.concat(
    {
        (f"{c:.0%}", arm): evaluate(c, **options)
        for c in (0.06, 0.08, 0.10, 0.12, 0.14)
        for arm, options in arms.items()
    },
    axis=1,
).T
grid.index.names = ["ceiling / target", "policy"]
grid[["sharpe", "vol_ann", "calibration", "max_drawdown", "turnover_ann", "mean_cost_ann",
      "mean_forecast_vol", "mean_gross", "share_levered_up"]]

sharpe  vol_ann  calibration  \
ceiling / target policy                                            
6%               limit (gross 3x)    1.164    0.073        1.271   
                 target (gross 3x)   1.164    0.073        1.271   
                 target (gross 6x)   1.173    0.073        1.272   
8%               limit (gross 3x)    1.124    0.093        1.272   
                 target (gross 3x)   1.124    0.093        1.272   
                 target (gross 6x)   1.173    0.098        1.272   
10%              limit (gross 3x)    1.092    0.104        1.272   
                 target (gross 3x)   1.098    0.106        1.270   
                 target (gross 6x)   1.172    0.122        1.272   
12%              limit (gross 3x)    1.071    0.110        1.274   
                 target (gross 3x)   1.072    0.118        1.270   
                 target (gross 6x)   1.166    0.145        1.273   
14%              limit (gross 3x)    1.063    0.111        1.274   
                 target (gross 3x)   1.041    0.127        1.270   
                 target (gross 6x)   1.143    0.167        1.274   

                                    max_drawdown  turnover_ann  mean_cost_ann  \
ceiling / target policy                                                         
6%               limit (gross 3x)         -0.092         6.072          0.001   
                 target (gross 3x)        -0.092         6.072          0.001   
                 target (gross 6x)        -0.092         6.371          0.001   
8%               limit (gross 3x)         -0.120         7.028          0.001   
                 target (gross 3x)        -0.120         7.059          0.001   
                 target (gross 6x)        -0.121         8.478          0.002   
10%              limit (gross 3x)         -0.144         7.408          0.001   
                 target (gross 3x)        -0.144         7.697          0.002   
                 target (gross 6x)        -0.150        10.504          0.002   
12%              limit (gross 3x)         -0.171         7.514          0.002   
                 target (gross 3x)        -0.171         8.203          0.002   
                 target (gross 6x)        -0.178        12.118          0.002   
14%              limit (gross 3x)         -0.190         7.536          0.002   
                 target (gross 3x)        -0.191         8.571          0.002   
                 target (gross 6x)        -0.205        13.325          0.003   

                                    mean_forecast_vol  mean_gross  \
ceiling / target policy                                             
6%               limit (gross 3x)               0.057       1.817   
                 target (gross 3x)              0.057       1.817   
                 target (gross 6x)              0.058       1.845   
8%               limit (gross 3x)               0.072       2.214   
                 target (gross 3x)              0.072       2.218   
                 target (gross 6x)              0.077       2.457   
10%              limit (gross 3x)               0.080       2.381   
                 target (gross 3x)              0.082       2.425   
                 target (gross 6x)              0.096       3.062   
12%              limit (gross 3x)               0.084       2.442   
                 target (gross 3x)              0.090       2.574   
                 target (gross 6x)              0.114       3.615   
14%              limit (gross 3x)               0.085       2.457   
                 target (gross 3x)              0.096       2.679   
                 target (gross 6x)              0.130       4.073   

                                    share_levered_up  
ceiling / target policy                               
6%               limit (gross 3x)              0.000  
                 target (gross 3x)             0.000  
                 target (gross 6x)             0.004  
8%               limit (gross 3x)              0.000

- **Target at 3x gross is almost the limit** (Sharpe 1.098 against 1.092): the gross cap blocks levering up.
- **Target at 6x gross, 10%:** levers up in 66% of months. Sharpe 1.172, vol 12.2%, turnover 10.5x (limit 7.4x), drawdown -15.0% (limit -14.4%). Its Sharpe stays 1.14-1.17 at every level, while the limit's falls from 1.16 (6%) to 1.06 (14%).
- **Nearest matched risk:** 6x target at 8% (vol 9.8%, Sharpe 1.173, turnover 8.5x) against the limit at 10% (vol 10.4%, Sharpe 1.092, turnover 7.4x): about +0.08 Sharpe for one more turn of turnover.
- **Calibration is 1.27 in every arm.** Scaling by the forecast does not fix the forecast.
- Single in-sample path; the Sharpe gaps are untested and the 15 variants add to the trial count.

## AFML improvements

AFML (Ch. 10, 14, 16) does not cover volatility targeting, leverage or gross caps. Two things carry over:

- **Deflated Sharpe (14.7.3):** the 15 variants above are trials.
- **Out-of-sample design (16.6):** compare scalings across many resampled paths, not one history.

## Suggested changes and experiments

| # | Experiment | Why |
|---|---|---|
| S1 | Paired Sharpe difference, limit against target at matched realised vol (`paired_sharpe_difference`) | The +0.08 is untested. |
| S2 | Sweep the gross cap (3x, 4x, 6x) under the target | Tests whether the gross cap is the real constraint. |
| S3 | Repeat on the configured model (DCC-GARCH with rescale) | Ceiling strictness depends on calibration. |
| S4 | Tail check before any copula or VaR/ES change: share of standardised returns below -2σ and -3σ against 2.3% and 0.13% | If tails are Gaussian and the std is above 1, fix the level (rescale). If tails are heavy, consider a tail measure in place of `sqrt(w'Qw)`. |